# 🔄 The Pivot — GRPO Training on Google Colab
**Meta PyTorch OpenEnv Hackathon 2026**

Trains **Qwen2.5-0.5B-Instruct** with GRPO + Adaptive Curriculum on the Pivot environment.

**Before you start:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — Check GPU

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — Runtime → Change runtime type → T4')
print(f'PyTorch CUDA: {torch.cuda.is_available()}')

## Cell 2 — Install packages (~3 min, run once per session)

In [ ]:
%%capture
!pip install -q openenv-core
!pip install -q "transformers>=4.45.0"
!pip install -q "peft>=0.13.0"
!pip install -q "bitsandbytes>=0.43.0"
!pip install -q "accelerate>=0.34.0"
!pip install -q wandb pydantic numpy python-dotenv
print('done')

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Model will save to: {SAVE_DIR}')

## Cell 4 — Clone project from GitHub

In [ ]:
import os, sys
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler

PROJECT_ROOT = '/content/meta_scaler'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# NOTE: Only disable W&B inside the RL environment (stops env from creating its own runs).
# We keep W&B active for the training loop — do NOT set WANDB_MODE=disabled globally.
os.environ['THE_PIVOT_WANDB_DISABLED'] = '1'

try:
    from models import PivotAction, ActionType, PivotObservation
    from server.pivot_environment import ThePivotEnvironment
    from server.prompt_encoder import encode_to_messages
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports OK!')
except ImportError as e:
    print(f'❌ {e}')

## Cell 5 — W&B Login

In [ ]:
import wandb
wandb.login()
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

## Cell 6 — Load model
Qwen2.5-0.5B with 4-bit quantization. Uses ~0.5 GB VRAM — leaves 14+ GB free for training.

In [ ]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME     = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE         = 'cuda'
MAX_SEQ_LEN    = 512
MAX_NEW_TOKENS = 8

print(f'Loading {MODEL_NAME}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
)
model = get_peft_model(base_model, lora_config)

# Critical: allows gradients to flow from frozen base layers into LoRA adapters
model.enable_input_require_grads()

# Verify LoRA is trainable
model.print_trainable_parameters()
trainable_count = sum(1 for p in model.parameters() if p.requires_grad)
print(f'Param groups with requires_grad=True: {trainable_count}')

gc.collect()
torch.cuda.empty_cache()
used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model ready! GPU: {used:.2f} GB used / {total:.1f} GB total  ({total-used:.1f} GB free)')

## Cell 7 — Helpers (generate, rollout, log-prob)

In [ ]:
import re, random, gc, numpy as np
from models import PivotAction, ActionType, PivotObservation
from server.pivot_environment import ThePivotEnvironment
from server.prompt_encoder import encode_to_messages

ACTION_MAP = {
    'execute':   ActionType.EXECUTE,
    'pivot':     ActionType.PIVOT,
    'research':  ActionType.RESEARCH,
    'fundraise': ActionType.FUNDRAISE,
    'hire':      ActionType.HIRE,
    'cut_costs': ActionType.CUT_COSTS,
    'cut':       ActionType.CUT_COSTS,
}

def _parse(text: str) -> ActionType:
    w = re.sub(r'[^a-z_]', '', text.lower().split()[0]) if text.strip() else 'execute'
    return ACTION_MAP.get(w, ActionType.EXECUTE)


@torch.no_grad()
def generate_action(obs: PivotObservation) -> tuple[str, ActionType]:
    messages = encode_to_messages(obs)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp  = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    out  = model.generate(
        **inp, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    del inp, out
    return decoded, _parse(decoded)


def get_log_prob(prompt: str, completion: str) -> torch.Tensor:
    """
    Log prob of completion tokens given prompt.
    FIXED: tokenizes completion separately, truncates prompt from LEFT
    so completion tokens always survive — previous bug was prompt filling
    all 512 tokens and completion getting cut off entirely (loss = 0).
    """
    # Step 1: tokenize completion alone — never truncate this
    comp_ids_cpu = tokenizer(
        completion, return_tensors='pt', add_special_tokens=False
    )['input_ids'][0]
    comp_len = comp_ids_cpu.shape[0]

    if comp_len == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)

    # Step 2: truncate prompt from LEFT to leave room for completion
    max_prompt_len = MAX_SEQ_LEN - comp_len
    prompt_ids_cpu = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=max_prompt_len
    )['input_ids'][0]

    # Step 3: combine [prompt | completion] and move to GPU
    full_ids = torch.cat([prompt_ids_cpu, comp_ids_cpu]).unsqueeze(0).to(DEVICE)
    p_len    = prompt_ids_cpu.shape[0]

    # Step 4: forward pass with gradients
    out         = model(input_ids=full_ids)
    comp_logits = out.logits[0, p_len - 1:-1, :].clone()
    target_ids  = full_ids[0, p_len:].clone()
    del out, full_ids
    torch.cuda.empty_cache()

    # Step 5: log softmax over completion tokens
    lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
    return lp[range(len(target_ids)), target_ids].mean()


def run_episode(scenario: dict) -> list[dict]:
    env  = ThePivotEnvironment(scenario=scenario)
    obs  = env.reset()
    steps = []
    for _ in range(60):
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True
        )
        completion, action_type = generate_action(obs)
        next_obs = env.step(PivotAction(action_type=action_type))
        steps.append({
            'prompt':     prompt,
            'completion': completion,
            'action':     action_type.value,
            'reward':     float(next_obs.reward or 0),
            'step':       next_obs.step,
            'runway':     next_obs.runway_remaining,
        })
        obs = next_obs
        if obs.done:
            break
    return steps


# Sanity check — run this after Cell 6 to verify gradients flow
def verify_gradients():
    model.train()
    from server.pivot_environment import ThePivotEnvironment
    env  = ThePivotEnvironment()
    obs  = env.reset()
    prompt = tokenizer.apply_chat_template(
        encode_to_messages(obs), tokenize=False, add_generation_prompt=True
    )
    completion = 'EXECUTE'
    lp = get_log_prob(prompt, completion)
    print(f'log_prob : {lp.item():.4f}')
    print(f'requires_grad: {lp.requires_grad}   ← must be True')
    print(f'grad_fn  : {lp.grad_fn}   ← must NOT be None')
    model.eval()

verify_gradients()
print('✅ Helpers ready!')

## Cell 8 — Config + GRPO update

In [ ]:
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER

CONFIG = {
    'n_episodes':        150,
    'G':                 2,    # 2 completions per step
    'lr':                5e-5,
    'grad_clip':         1.0,
    'grpo_steps_sample': 3,
    'save_every':        25,
    'log_every':         5,
}

optimizer  = AdamW(model.parameters(), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)


def grpo_update(episode_steps: list[dict]) -> dict:
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0

    sampled = random.sample(episode_steps,
                            min(CONFIG['grpo_steps_sample'], len(episode_steps)))

    for sd in sampled:
        prompt      = sd['prompt']
        completions = [sd['completion']]
        rewards     = [sd['reward']]

        # One alternative completion
        for _ in range(CONFIG['G'] - 1):
            with torch.no_grad():
                inp = tokenizer(prompt, return_tensors='pt',
                                truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
                out = model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                     do_sample=True, temperature=1.1,
                                     pad_token_id=tokenizer.eos_token_id)
                alt = tokenizer.decode(out[0][inp['input_ids'].shape[1]:],
                                       skip_special_tokens=True).strip()
                del inp, out
                torch.cuda.empty_cache()
            completions.append(alt)
            rewards.append(sd['reward'] + random.gauss(0, 1.5))

        # Normalise within group
        r_arr  = np.array(rewards, dtype=np.float32)
        normed = (r_arr - r_arr.mean()) / (r_arr.std() + 1e-8)

        # Policy gradient
        for comp, nr in zip(completions, normed):
            lp        = get_log_prob(prompt, comp)
            step_loss = (-lp * float(nr)) / (CONFIG['G'] * CONFIG['grpo_steps_sample'])
            step_loss.backward()
            total_loss += step_loss.item()
            del lp, step_loss
            gc.collect()
            torch.cuda.empty_cache()

    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
    optimizer.step()
    optimizer.zero_grad()
    model.eval()
    gc.collect()
    torch.cuda.empty_cache()

    return {'loss': total_loss}


print('✅ Config + GRPO ready!')
print('Config:', CONFIG)

## Cell 9 — Train! (~60-90 min on T4)

In [ ]:
# Make sure W&B is active for training (not disabled)
import os
os.environ.pop('WANDB_MODE', None)
os.environ.pop('THE_PIVOT_WANDB_DISABLED', None)

run = wandb.init(
    project=WANDB_PROJECT,
    name='grpo-qwen0.5b-pivot',
    config=CONFIG,
    tags=['grpo', 'qwen0.5b', 'pivot', 'hackathon'],
)
print(f'W&B run: {run.get_url()}')
print(f'Starting tier 1: {DIFFICULTY_LADDER[curriculum.current_tier]}')
print('─' * 70)

episode_rewards, episode_lengths, all_losses = [], [], []
model.eval()

for ep in range(1, CONFIG['n_episodes'] + 1):

    scenario      = curriculum.sample_scenario()
    scenario_name = scenario.get('name', 'default')
    steps         = run_episode(scenario)

    ep_reward   = sum(s['reward'] for s in steps)
    ep_length   = len(steps)
    survived    = ep_length >= 60
    pivot_count = sum(1 for s in steps if s['action'] == 'PIVOT')

    episode_rewards.append(ep_reward)
    episode_lengths.append(ep_length)

    loss_stats = grpo_update(steps)
    all_losses.append(loss_stats['loss'])
    curriculum.record_result(ep_reward, survived)

    gpu_gb = torch.cuda.memory_allocated() / 1e9
    print(
        f'Ep {ep:3d}/{CONFIG["n_episodes"]} | {scenario_name:18s} | '
        f'T{curriculum.current_tier+1} | steps {ep_length:3d} | '
        f'reward {ep_reward:+7.1f} | loss {loss_stats["loss"]:7.4f} | '
        f'pivots {pivot_count} | {"SURVIVED" if survived else "died"} | '
        f'GPU {gpu_gb:.1f}GB'
    )

    if ep % CONFIG['log_every'] == 0:
        wandb.log({
            'episode':          ep,
            'ep_reward':        ep_reward,
            'mean_reward_20ep': float(np.mean(episode_rewards[-20:])),
            'ep_length':        ep_length,
            'survived':         int(survived),
            'pivot_count':      pivot_count,
            'loss':             loss_stats['loss'],
            'curriculum_tier':  curriculum.current_tier + 1,
            'gpu_gb':           gpu_gb,
            'scenario':         scenario_name,
        })

    if curriculum.should_advance() and curriculum.advance_tier():
        new = DIFFICULTY_LADDER[curriculum.current_tier]
        print(f'\n🎓 CURRICULUM → Tier {curriculum.current_tier+1}/5: {new}\n')
        wandb.log({'curriculum_advance': curriculum.current_tier, 'episode': ep})

    if ep % CONFIG['save_every'] == 0:
        ckpt = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        print(f'💾 Checkpoint → {ckpt}')


print('\n' + '='*70)
print('✅ TRAINING COMPLETE!')
print(f'Mean reward last 20 ep : {np.mean(episode_rewards[-20:]):.1f}')
print(f'Best reward            : {max(episode_rewards):.1f}')
print(f'Survival rate          : {sum(l>=60 for l in episode_lengths)/len(episode_lengths):.0%}')
print('='*70)

## Cell 10 — Save final model + close W&B

In [ ]:
FINAL = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
print(f'✅ Saved to {FINAL}')
wandb.finish()
print('✅ W&B closed')

## Cell 11 — Evaluate: trained vs baselines

In [ ]:
from training.baseline_agent import RandomAgent, StubbornAgent, PanicAgent, run_episodes
import json

N_EVAL = 20
c = AdaptiveCurriculum()

class TrainedAgent:
    name = 'trained_llm'
    def act(self, obs):
        _, at = generate_action(obs)
        return PivotAction(action_type=at)

agents = [RandomAgent(), StubbornAgent(), PanicAgent(), TrainedAgent()]
print(f'{"Scenario":22s} | {"Agent":12s} | Reward   | Survival | PivotRate')
print('-' * 68)
all_results = []
for name in DIFFICULTY_LADDER:
    sc = c._all_scenarios.get(name)
    if not sc: continue
    for agent in agents:
        r = run_episodes(agent, sc, N_EVAL)
        all_results.append(r)
        print(f'{name:22s} | {agent.name:12s} | {r["mean_reward"]:+7.1f} | {r["survival_rate"]:6.0%}   | {r["pivot_rate"]:6.0%}')

with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\n✅ Results saved to Drive')